In [27]:
from preprocessing import load_GSE127465, load_nsclc_cosmx
# from transcript_sampling import add_top_st_matches_with_coords
from mapping import integrate_single_st_file_with_harmony
import anndata as ad
import numpy as np
import pandas as pd
from label_transfer import preprocess_and_transfer_cell_types, make_label_correspondence_table, plot_label_correspondence_heatmap
import matplotlib.pyplot as plt
import os

spatial_dir = '/home/jwinchel/Syndri/data/breast_cancer_rctd_by_sample/'

# sc_data = load_GSE127465('/compute/oven-0-13/jwinchel/scRNA-seq',)
sc_data = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/split_sc_ref/breast_cancer_chromium_genesubset.h5ad')
# st_data = ad.read_h5ad('/compute/oven-0-13/jwinchel/split_spatial_chromium/split_xenium/GSM9509172_breast_breast_B1_B1_1_rctd_277genes.h5ad')
# sc_data = sc_data[sc_data.obs['Tissue']=='tumor'] # only tumor data

In [ ]:
# spatial_files = sorted([f for f in os.listdir(spatial_dir) if f.endswith('rctd.h5ad')])
# save_dir = '/compute/oven-0-13/jwinchel/split_spatial_chromium/split_xenium/'
# # restrict to intersection of gene sets
# overlap = sorted(list(set(sc_data.var_names).intersection(set(st_data.var['gene_ids'].unique()))))
# for f in spatial_files:
#     spat = ad.read_h5ad(os.path.join(spatial_dir,f))

#     gene_mask = spat.var_names.isin(overlap)
#     spat = spat[:,gene_mask].copy()
#     sorted_idx = spat.var_names.argsort()
#     spat = spat[:,sorted_idx].copy()
#     f_subset = f.split('.')[0] + '_277genes.h5ad'
#     spat.write_h5ad(os.path.join(save_dir,f_subset))

# gene_mask = sc_data.var_names.isin(overlap)
# chrom_subset = sc_data[:,gene_mask].copy()
# sorted_idx = chrom_subset.var_names.argsort()
# chrom_subset = chrom_subset[:,sorted_idx].copy()
# chrom_subset.write_h5ad(os.path.join(save_dir,'breast_cancer_chromium_277genes.h5ad'))

In [28]:
good_mask = (2000 >= sc_data.X.sum(axis=1)) & (sc_data.X.sum(axis=1) >= 10)
sc_data = sc_data[good_mask]

In [29]:
cell_types = sorted(sc_data.obs['cell_type'].unique())
cell_type_dict = {cell_types[i]: -(i+1) for i in range(len(cell_types))}

sc_data.uns['cell_type_dict'] = cell_type_dict

# sc_data.obs['split'] = 'train'
# sc_data.obs.loc[sc_data.obs['donor_id']=='1','split'] = 'val'

/tmp/ipykernel_1204910/3124072026.py:4: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  sc_data.uns['cell_type_dict'] = cell_type_dict


In [30]:
import os
import importlib
import mapping
importlib.reload(mapping)
from mapping import integrate_single_st_file_with_harmony
save_dir = '/compute/oven-0-13/jwinchel/split_spatial_chromium/split_xenium/'
spatial_files = sorted([f for f in os.listdir(spatial_dir) if f.endswith('rctd.h5ad')])
# spatial_files = ['GSM9509174_breast_breast_B1_B1_3_rctd.h5ad']
print(spatial_files)
for f in spatial_files:
    tissue_name = f.split('_')[0]

    print(tissue_name)


    if f'X_pca_harmony' in sc_data.obsm:
        continue
    # batch integration
    sc_data, st_data, cov =  integrate_single_st_file_with_harmony(st_file=os.path.join(spatial_dir, f), sc_data_orig=sc_data, tissue_filter=None)

    st_data.write_h5ad(os.path.join(save_dir, f))
    
    print(sc_data.obsm)



['GSM9509156_breast_breast_B17_B17_rctd.h5ad', 'GSM9509157_breast_breast_B16_B16_rctd.h5ad', 'GSM9509158_breast_breast_B15_B15_rctd.h5ad', 'GSM9509159_breast_breast_B14_B14_rctd.h5ad', 'GSM9509160_breast_breast_B2_B2_rctd.h5ad', 'GSM9509161_breast_breast_B13_B13_rctd.h5ad', 'GSM9509162_breast_breast_B12_B12_rctd.h5ad', 'GSM9509163_breast_breast_B11_B11_rctd.h5ad', 'GSM9509164_breast_breast_B10_B10_rctd.h5ad', 'GSM9509165_breast_breast_B4_B4_rctd.h5ad', 'GSM9509166_breast_breast_B9_B9_rctd.h5ad', 'GSM9509167_breast_breast_B8_B8_rctd.h5ad', 'GSM9509168_breast_breast_B7_B7_rctd.h5ad', 'GSM9509169_breast_breast_B6_B6_rctd.h5ad', 'GSM9509170_breast_breast_B3_B3_rctd.h5ad', 'GSM9509171_breast_breast_B5_B5_rctd.h5ad', 'GSM9509172_breast_breast_B1_B1_1_rctd.h5ad', 'GSM9509173_breast_breast_B1_B1_2_rctd.h5ad', 'GSM9509174_breast_breast_B1_B1_3_rctd.h5ad']
GSM9509156


/home/jwinchel/Syndri/code/mapping.py:64: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
2026-06-16 17:03:56,972 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample_id', 'percent_mt', 'percent_ribo', 'percent_hb', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.0.5', 'SCT_snn_res.0.4', 'SCT_snn_res.2', 'donor_id', 'nCount_SoupX', 'nFeature_SoupX', 'Harmonised_Level4', 'Level3', 'Level2', 'Level1', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'input_material', 'histology', 'sample_type', 'histology_immune', 'DV200_percent', 'block_age_months', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'batch', 'cell_id', 'cells_parquet_matched', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecate

2026-06-16 17:04:03,634 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-06-16 17:04:03,806 - harmonypy - INFO - Iteration 1 of 30
2026-06-16 17:04:17,998 - harmonypy - INFO - Iteration 2 of 30
2026-06-16 17:04:31,901 - harmonypy - INFO - Iteration 3 of 30
2026-06-16 17:04:45,759 - harmonypy - INFO - Iteration 4 of 30
2026-06-16 17:05:00,060 - harmonypy - INFO - Iteration 5 of 30
2026-06-16 17:05:14,560 - harmonypy - INFO - Iteration 6 of 30
2026-06-16 17:05:28,938 - harmonypy - INFO - Iteration 7 of 30
2026-06-16 17:05:36,988 - harmonypy - INFO - Iteration 8 of 30
2026-06-16 17:05:41,489 - harmonypy - INFO - Iteration 9 of 30
2026-06-16 17:05:46,592 - harmonypy - INFO - Converged after 9 iterations


AxisArrays with keys: X_umap, X_pca_harmony_GSM9509156_breast_breast_B17_B17_rctd
GSM9509157


/home/jwinchel/Syndri/code/mapping.py:64: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
2026-06-16 17:05:48,698 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


... (67 outputs left out)

/home/jwinchel/Syndri/code/mapping.py:64: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata = sc_data.concatenate(
2026-06-16 17:58:04,012 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...


['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample_id', 'percent_mt', 'percent_ribo', 'percent_hb', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.0.5', 'SCT_snn_res.0.4', 'SCT_snn_res.2', 'donor_id', 'nCount_SoupX', 'nFeature_SoupX', 'Harmonised_Level4', 'Level3', 'Level2', 'Level1', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'input_material', 'histology', 'sample_type', 'histology_immune', 'DV200_percent', 'block_age_months', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'batch', 'cell_id', 'cells_parquet_matched', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecate

2026-06-16 17:58:12,547 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-06-16 17:58:12,801 - harmonypy - INFO - Iteration 1 of 30
2026-06-16 17:58:37,900 - harmonypy - INFO - Iteration 2 of 30
2026-06-16 17:59:03,014 - harmonypy - INFO - Iteration 3 of 30
2026-06-16 17:59:28,631 - harmonypy - INFO - Iteration 4 of 30
2026-06-16 17:59:54,988 - harmonypy - INFO - Iteration 5 of 30
2026-06-16 18:00:18,062 - harmonypy - INFO - Iteration 6 of 30
2026-06-16 18:00:33,931 - harmonypy - INFO - Iteration 7 of 30
2026-06-16 18:00:46,034 - harmonypy - INFO - Iteration 8 of 30
2026-06-16 18:00:56,756 - harmonypy - INFO - Iteration 9 of 30
2026-06-16 18:01:22,399 - harmonypy - INFO - Iteration 10 of 30
2026-06-16 18:01:48,175 - harmonypy - INFO - Iteration 11 of 30
2026-06-16 18:01:57,595 - harmonypy - INFO - Iteration 12 of 30
2026-06-16 18:02:07,006 - harmonypy - INFO - Iteration 13 of 30
2026-06-16 18:02:15,406 - harmonypy - INFO - Iteration 14 of 30
2026-06-16 18:02:25,376 - ha

AxisArrays with keys: X_umap, X_pca_harmony_GSM9509156_breast_breast_B17_B17_rctd, X_pca_harmony_GSM9509157_breast_breast_B16_B16_rctd, X_pca_harmony_GSM9509158_breast_breast_B15_B15_rctd, X_pca_harmony_GSM9509159_breast_breast_B14_B14_rctd, X_pca_harmony_GSM9509160_breast_breast_B2_B2_rctd, X_pca_harmony_GSM9509161_breast_breast_B13_B13_rctd, X_pca_harmony_GSM9509162_breast_breast_B12_B12_rctd, X_pca_harmony_GSM9509163_breast_breast_B11_B11_rctd, X_pca_harmony_GSM9509164_breast_breast_B10_B10_rctd, X_pca_harmony_GSM9509165_breast_breast_B4_B4_rctd, X_pca_harmony_GSM9509166_breast_breast_B9_B9_rctd, X_pca_harmony_GSM9509167_breast_breast_B8_B8_rctd, X_pca_harmony_GSM9509168_breast_breast_B7_B7_rctd, X_pca_harmony_GSM9509169_breast_breast_B6_B6_rctd, X_pca_harmony_GSM9509170_breast_breast_B3_B3_rctd, X_pca_harmony_GSM9509171_breast_breast_B5_B5_rctd, X_pca_harmony_GSM9509172_breast_breast_B1_B1_1_rctd, X_pca_harmony_GSM9509173_breast_breast_B1_B1_2_rctd, X_pca_harmony_GSM9509174_breast_

In [31]:
sc_data.write_h5ad('/compute/oven-0-13/jwinchel/split_spatial_chromium/breast_cancer_chromium_integrated.h5ad')